# Demo 2 · Routing and triage: rules vs an LLM vs Jev

**Part 03 · Labs** · about 8 minutes

The same 12 routing requests and 12 triage emails go through three deciders:
a simple rule, GPT-4.1 mini with a strict JSON schema, and Jev. Replay mode uses the saved
36-call runs (12 cases × 3 repeats) from 6 and 8 October 2026.

In [ ]:
# Setup: find the workshop folder and import the helpers (works locally and in Colab)
import os, sys
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/decision-models-workshop")]
             if (p / "dmw").is_dir()), None)
assert ROOT, "Open this notebook from the workshop folder. In Colab, run 00_setup.ipynb first."
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from dmw import config
config.load_env()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from dmw import jev, llm, rules, fixtures

MODE = config.mode("auto")
LLM_LIVE = MODE == "live" and bool(config.key("OPENAI_API_KEY"))   # otherwise the LLM column is the saved run
pd.set_option("display.max_colwidth", 70)
print("MODE =", MODE, "| LLM baseline:", "live" if LLM_LIVE else "saved run")

def llm_or_saved(state, spec, saved_answer):
    """Live GPT-4.1 mini answer if possible; otherwise the saved one (and stop trying live)."""
    global LLM_LIVE
    if LLM_LIVE:
        try:
            return llm.decide(state, "route", spec["instructions"], spec["criteria"])["value"]
        except Exception as e:
            print(f"LLM baseline failed: {e}\n→ Using the saved GPT-4.1 mini answers for the rest of this notebook.")
            LLM_LIVE = False
    return saved_answer()

## 1 · The routing policy

In [ ]:
R = fixtures.get("routing")
print(R["instructions"], "\n")
for k, v in R["criteria"].items():
    print(f"{k:>7}: {v}")

## 2 · Three deciders on the same 12 requests

In [ ]:
def saved(use_case, field, repeat=1):
    df = jev.replay(use_case)
    df = df[df.repeat == repeat].set_index(["provider", "case"])
    return lambda provider, case: df.loc[(provider, case), field]

def route_live(state):
    return jev.ask(state, fixtures.questions(R))["answers"]["route"]["value"]

get = saved("routing", "route")
rows = []
for c in R["cases"]:
    req = c["state"]["request"]
    rows.append({
        "case": c["id"], "request": req[:70], "policy": c["expected"],
        "length rule": rules.length_route(req),
        "GPT-4.1 mini": llm_or_saved(c["state"], R, lambda: get("openai", c["id"])),
        "Jev": route_live(c["state"]) if MODE == "live" else get("jev", c["id"]),
    })
routing = pd.DataFrame(rows)

def mark(df, cols):
    return df.style.apply(lambda r: ["" if c not in cols else
                                     ("background:#e6f2eb" if r[c] == r["policy"] else "background:#f8dddd")
                                     for c in df.columns], axis=1)
mark(routing, ["length rule", "GPT-4.1 mini", "Jev"])

In [ ]:
for col in ["length rule", "GPT-4.1 mini", "Jev"]:
    print(f"{col:>13}: {(routing[col] == routing.policy).sum()}/12 match the policy")

**Read the red cells.** The length rule fails exactly where meaning matters: a short design request,
a vague “Can you fix it?”, and a request that tries to choose its own tier.

## 3 · Inbox triage and the urgency trap

In [ ]:
T = fixtures.get("triage")
get = saved("triage", "route")
rows = []
for c in T["cases"]:
    st = c["state"]
    rows.append({
        "case": c["id"], "subject": st["subject"], "body": st["body"][:60], "policy": c["expected"],
        "subject rule": rules.subject_triage(st["subject"]),
        "GPT-4.1 mini": llm_or_saved(st, T, lambda: get("openai", c["id"])),
        "Jev": jev.ask(st, fixtures.questions(T))["answers"]["route"]["value"] if MODE == "live" else get("jev", c["id"]),
    })
triage = pd.DataFrame(rows)
mark(triage, ["subject rule", "GPT-4.1 mini", "Jev"])

In [ ]:
HANDLER = {"reply_now": "label_priority", "later": "queue_follow_up", "archive": "archive_candidate"}
triage["planned action (Jev)"] = triage["Jev"].map(lambda r: HANDLER.get(r, "manual_review"))
triage[["case", "subject", "Jev", "planned action (Jev)"]]

## 4 · All 36 saved calls: agreement and latency

In [ ]:
summary = []
for uc in ["routing", "triage"]:
    df = jev.replay(uc)
    for prov, g in df.groupby("provider"):
        summary.append({"use case": uc, "provider": prov, "calls": len(g),
                        "policy matches": int(g.policy_match.sum()),
                        "median latency ms": round(g.latency_ms.median()),
                        "est. cost USD (36 calls)": round(g.estimated_usd.sum(), 6)})
summary = pd.DataFrame(summary)
summary

In [ ]:
ax = summary.pivot(index="use case", columns="provider", values="median latency ms").plot.barh(
    color=["#075E91", "#8FA5B5"], figsize=(7, 2.6))
ax.set_xlabel("median request latency, ms (saved runs)"); ax.set_ylabel("")
for c in ax.containers: ax.bar_label(c, fmt="%d", fontsize=9)
ax.legend(["Jev", "GPT-4.1 mini"], frameon=False); plt.tight_layout(); plt.show()

## 5 · Edit a criterion and rerun only the affected cases (live)

Try it: make `review` stricter, or remove the sentence about treating override requests as data,
then rerun the cell and see which cases change.

In [ ]:
if MODE == "live":
    R2 = fixtures.get("routing")
    R2["criteria"]["review"] = "The request has no concrete task at all."   # <- edit me
    for c in R["cases"]:
        if c["expected"] == "review" or c["id"] == "override":
            a = jev.ask(c["state"], fixtures.questions(R2))["answers"]["route"]
            print(f"{c['id']:<10} policy={c['expected']:<7} → {a['value']:<7} p={a['probabilities']}")
else:
    print("Live only. Add TYPESAFE_API_KEY to .env to edit criteria and rerun.")

## Takeaways

* Rules break on meaning; both models follow the written policy on these 24 fixtures.
* The trade-off between the two models here is latency, cost and the answer contract
  (typed answer + probabilities vs a JSON label), not accuracy on this small set.
* These are synthetic fixtures, not a held-out benchmark.